In [65]:
"""
Titanic Data Analysis and JSON Export
Author: [Markus von Aschoff]
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""
 
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data/titanic.csv


Step 2

In [66]:
# Load the dataset
df = pd.read_csv(CSV_FILE)

print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()


Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Calculating Statistics (Step 3)

In [67]:
print("=" * 55)
print("DESCRIPTIVE STATISTICS")
print("=" * 55)

# Select numeric columns only
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_columns = [c for c in numeric_columns if c not in ["PassengerId", "Survived"]]

# Calculate statistics (.mean, median, std)
stats_rows = []
for col in numeric_columns:
    stats_rows.append({
        "Column":   col,
        "Mean":     round(df[col].mean(),   4),
        "Median":   round(df[col].median(), 4),
        "Std Dev":  round(df[col].std(),    4),
        "Min":      round(df[col].min(),    4),
        "Max":      round(df[col].max(),    4),
    })

stats_df = pd.DataFrame(stats_rows).set_index("Column")
print(stats_df.to_string())
stats_df

DESCRIPTIVE STATISTICS
           Mean   Median  Std Dev   Min       Max
Column                                           
Pclass   2.3086   3.0000   0.8361  1.00    3.0000
Age     29.6991  28.0000  14.5265  0.42   80.0000
SibSp    0.5230   0.0000   1.1027  0.00    8.0000
Parch    0.3816   0.0000   0.8061  0.00    6.0000
Fare    32.2042  14.4542  49.6934  0.00  512.3292


,Mean,Median,Std Dev,Min,Max
Column,,,,,
Pclass,2.3086,3.0000,0.8361,1.00,3.0000
Age,29.6991,28.0000,14.5265,0.42,80.0000
SibSp,0.5230,0.0000,1.1027,0.00,8.0000
Parch,0.3816,0.0000,0.8061,0.00,6.0000
Fare,32.2042,14.4542,49.6934,0.00,512.3292


### Step 4

In [68]:
print("=" * 55)
print("MISSING VALUES ANALYSIS")
print("=" * 55)

missing_data = {}

for col in df.columns:
    missing_count   = df[col].isna().sum()
    missing_percent = (missing_count / len(df)) * 100
    missing_data[col] = {
        "missing_count":   int(missing_count),
        "missing_percent": round(missing_percent, 2),
    }

# Build a readable DataFrame
missing_df = (
    pd.DataFrame(missing_data).T
    .rename(columns={"missing_count": "Missing", "missing_percent": "Missing %"})
    .sort_values("Missing", ascending=False)
)

#print(missing_df[missing_df["Missing"] > 0].to_string())
missing_df[missing_df["Missing"] > 0]

MISSING VALUES ANALYSIS


,Missing,Missing %
Cabin,687.0,77.10
Age,177.0,19.87
Embarked,2.0,0.22


### Step 5

In [69]:
# Create a copy of the dataframe for feature engineering
df_features = df.copy()
 
# Feature 1: Family Size
df_features["FamilySize"] = df_features["SibSp"] + df_features["Parch"] + 1
print("FamilySize sample:")
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))
 
# Feature 2: Is Alone
df_features["IsAlone"] = (df_features["FamilySize"] == 1).astype(int)
print("IsAlone sample:")
print(df_features[["FamilySize", "IsAlone"]].head(10))
 
# Feature 3: Age Groups
def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return "Unknown"
    elif age < 18:
        return "Child/Teen"
    elif age < 30:
        return "Young Adult"
    elif age < 50:
        return "Middle-Aged"
    else:
        return "Senior"
    
df_features["AgeGroup"] = df_features["Age"].apply(categorize_age)
print("AgeGroup sample:")
print(df_features[["Age", "AgeGroup"]].head(10))


# ── Feature 4: Title (extracted from Name) ────────────────────────────────
df_features["Title"] = df_features["Name"].str.extract(r",\s*([^\.]+)\.")

title_map = {
    "Mr":       "Mr",     "Miss":   "Miss",  "Mrs":    "Mrs",
    "Master":   "Master", "Dr":     "Officer","Rev":    "Officer",
    "Col":      "Officer","Major":  "Officer","Capt":   "Officer",
    "Don":      "Royalty","Lady":   "Royalty","Countess":"Royalty",
    "Mme":      "Mrs",    "Ms":     "Miss",   "Mlle":   "Miss",
    "Sir":      "Royalty","Jonkheer":"Royalty",
}

df_features["Title"] = df_features["Title"].map(title_map).fillna("Other")
print("Title distribution:")
print(df_features["Title"].value_counts())

# ── Feature 5: Fare Per Person ────────────────────────────────────────────
ticket_counts = df_features.groupby("Ticket")["PassengerId"].count()
df_features["TicketGroupSize"] = df_features["Ticket"].map(ticket_counts)
df_features["FarePerPerson"]   = (df_features["Fare"] / df_features["TicketGroupSize"]).round(4)
print(f"FarePerPerson – Mean: £{df_features['FarePerPerson'].mean():.2f}  "      f"Median: £{df_features['FarePerPerson'].median():.2f}")



 
# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50)
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)
 
# Here is an example to get you started:
print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)
 
# Statistical test: Do these features help differentiate?
print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)
 
survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]
 
print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

print("\nAgeGroup survival rate:")
print(df_features.groupby("AgeGroup")["Survived"].mean().sort_values(ascending=False).round(3))

FamilySize sample:
   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2
IsAlone sample:
   FamilySize  IsAlone
0           2        0
1           2        0
2           1        1
3           2        0
4           1        1
5           1        1
6           1        1
7           5        0
8           3        0
9           2        0
AgeGroup sample:
    Age     AgeGroup
0  22.0  Young Adult
1  38.0  Middle-Aged
2  26.0  Young Adult
3  35.0  Middle-Aged
4  35.0  Middle-Aged
5   NaN      Unknown
6  54.0       Senior
7   2.0   Child/Teen
8  27.0  Young Adult
9  14.0   Child/Teen
Title distribution:
Title
Mr         517
Miss       185
Mrs        126
Master      40
Officer     18
Royalty      4
Other        1
Name: count, dtype: int64

### Step 6 

In [73]:
# Step 3: Create Classes for JSON Export
 
class Passenger:
    """
    Represents a passenger with all their information.
    """
    def __init__(self, passenger_id, name, age, sex, survived, pclass, 
                 fare, embarked=None, family_size=None, is_alone=None, title=None, age_group=None, fare_per_person=None):
        # TODO: Initialize all passenger attributes
        # Tip: Use pd.notna() to check if a value is not null/NaN
        # Tip: Convert to appropriate types (int, float, str)
        # Example for passenger_id:
        self.passenger_id    = int(passenger_id)          if pd.notna(passenger_id)    else None
        self.name            = str(name)                  if pd.notna(name)            else None
        self.age             = round(float(age), 1)       if pd.notna(age)             else None
        self.sex             = str(sex)                   if pd.notna(sex)             else None
        self.survived        = int(survived)              if pd.notna(survived)        else None
        self.pclass          = int(pclass)                if pd.notna(pclass)          else None
        self.fare            = round(float(fare), 4)      if pd.notna(fare)            else None
        self.embarked        = str(embarked)              if pd.notna(embarked)        else None
        self.family_size     = int(family_size)           if pd.notna(family_size)     else None
        self.is_alone        = int(is_alone)              if pd.notna(is_alone)        else None
        self.title           = str(title)                 if pd.notna(title)           else None
        self.age_group       = str(age_group)             if pd.notna(age_group)       else None
        self.fare_per_person = round(float(fare_per_person), 4) if pd.notna(fare_per_person) else None
        
        # TODO: Complete initialization for remaining attributes
        # Remember to handle None/NaN values appropriately
        
    
    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""
        # TODO: Return a dictionary with all passenger attributes
        # Tip: Dictionary keys should match the attribute names
        return {
            'passenger_id': self.passenger_id,
            # TODO: Add all other attributes here
            "name":            self.name,
            "age":             self.age,
            "sex":             self.sex,
            "survived":        self.survived,
            "pclass":          self.pclass,
            "fare":            self.fare,
            "embarked":        self.embarked,
            "family_size":     self.family_size,
            "is_alone":        self.is_alone,
            "title":           self.title,
            "age_group":       self.age_group,
            "fare_per_person": self.fare_per_person,
        }
    def __repr__(self):
        return (f"<Passenger id={self.passenger_id} name='{self.name}' "                f"survived={self.survived} class={self.pclass}>")

class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []  # Will store Passenger objects
        self._create_passengers()
    
    def _create_passengers(self):
        """Create Passenger objects from dataframe."""
        # TODO: Iterate through dataframe rows and create Passenger objects
        # Tip: Use self.dataframe.iterrows() to loop through rows
        # Tip: Use row.get('ColumnName', default_value) to safely get values
        for idx, row in self.dataframe.iterrows():
            # TODO: Create a Passenger object with data from the row
            # Example of getting PassengerId:
            # passenger_id = row.get('PassengerId', idx)
            p = Passenger(
                passenger_id    = row.get("PassengerId"),
                name            = row.get("Name"),
                age             = row.get("Age"),
                sex             = row.get("Sex"),
                survived        = row.get("Survived"),
                pclass          = row.get("Pclass"),
                fare            = row.get("Fare"),
                embarked        = row.get("Embarked"),
                family_size     = row.get("FamilySize"),
                is_alone        = row.get("IsAlone"),
                title           = row.get("Title"),
                age_group       = row.get("AgeGroup"),
                fare_per_person = row.get("FarePerPerson"),
            )
            
            # TODO: Create the Passenger object and append to self.passengers
            self.passengers.append(p)
    
   # ── Public methods ───────────────────────────────────────────────────────

    def get_summary_stats(self):
        """Return a dictionary of summary statistics."""
        survived_list = [p for p in self.passengers if p.survived == 1]
        not_surv_list = [p for p in self.passengers if p.survived == 0]

        ages  = [p.age  for p in self.passengers if p.age  is not None]
        fares = [p.fare for p in self.passengers if p.fare is not None]

        return {
            "total_passengers":     len(self.passengers),
            "survived":             len(survived_list),
            "did_not_survive":      len(not_surv_list),
            "survival_rate_pct":    round(len(survived_list) / len(self.passengers) * 100, 2),
            "average_age":          round(sum(ages)  / len(ages),  2) if ages  else None,
            "average_fare":         round(sum(fares) / len(fares), 2) if fares else None,
            "passengers_with_cabin": sum(1 for p in self.passengers
                                          if self.dataframe.loc[
                                              self.dataframe["PassengerId"] == p.passenger_id,
                                              "Cabin"].values[0] is not None
                                             and str(self.dataframe.loc[
                                              self.dataframe["PassengerId"] == p.passenger_id,
                                              "Cabin"].values[0]) != "nan"),
        }

    def get_missing_values(self):
        """Return missing-value counts from the original DataFrame."""
        result = {}
        for col in self.dataframe.columns:
            count = int(self.dataframe[col].isna().sum())
            if count > 0:
                result[col] = {
                    "missing_count":   count,
                    "missing_percent": round(count / len(self.dataframe) * 100, 2),
                }
        return result

    def to_json(self, filename="titanic_data.json"):
        """Serialise the full dataset (metadata + passengers) to JSON."""
        summary = self.get_summary_stats()

        data = {
            "metadata": {
                "dataset_name":        "Titanic Passenger Dataset",
                "export_date":         datetime.now().isoformat(),
                "total_passengers":    summary["total_passengers"],
                "survived":            summary["survived"],
                "did_not_survive":     summary["did_not_survive"],
                "survival_rate_pct":   summary["survival_rate_pct"],
                "average_age":         summary["average_age"],
                "average_fare":        summary["average_fare"],
                "missing_values":      self.get_missing_values(),
                "engineered_features": ["FamilySize", "IsAlone", "AgeGroup",
                                         "Title", "FarePerPerson"],
            },
            "passengers": [p.to_dict() for p in self.passengers],
        }

        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        print(f"Data exported to {filename}  ({Path(filename).stat().st_size / 1024:.1f} KB)")
        return data

    def __len__(self):
        return len(self.passengers)

    def __repr__(self):
        return f"<TitanicDataset passengers={len(self.passengers)}>"

In [74]:
# Ensure df_features exists (re-run Step 5 cells if needed)
assert "FamilySize" in df_features.columns, "Run Step 5 cells first!"

# Create the dataset object
dataset = TitanicDataset(df_features)
print(dataset)

# Display summary statistics
print("\n" + "=" * 55)
print("SUMMARY STATISTICS")
print("=" * 55)
for key, value in dataset.get_summary_stats().items():
    print(f"  {key:<28}: {value}")

# Export to JSON
json_data = dataset.to_json(str(JSON_FILE))


<TitanicDataset passengers=891>

SUMMARY STATISTICS
  total_passengers            : 891
  survived                    : 342
  did_not_survive             : 549
  survival_rate_pct           : 38.38
  average_age                 : 29.7
  average_fare                : 32.2
  passengers_with_cabin       : 204
Data exported to data/titanic_data.json  (296.8 KB)


### Step 7

In [75]:
# Load the JSON file back in
with open(JSON_FILE, "r", encoding="utf-8") as f:
    loaded = json.load(f)

print("=" * 55)
print("JSON VALIDATION")
print("=" * 55)

# ── Structure checks ────────────────────────────────────────────────────────
required_top_keys = {"metadata", "passengers"}
assert required_top_keys.issubset(loaded.keys()), f"Missing top-level keys: {required_top_keys - loaded.keys()}"
print(f"[PASS] Top-level keys present: {list(loaded.keys())}")

required_meta_keys = {"dataset_name","export_date","total_passengers",
                      "survived","did_not_survive","survival_rate_pct",
                      "average_age","average_fare","missing_values","engineered_features"}
assert required_meta_keys.issubset(loaded["metadata"].keys()), "Metadata incomplete"
print(f"[PASS] All required metadata keys present")

# ── Record count ────────────────────────────────────────────────────────────
record_count = len(loaded["passengers"])
assert record_count == 891, f"Expected 891 records, got {record_count}"
print(f"[PASS] Passenger record count: {record_count}")

# ── Passenger key check ─────────────────────────────────────────────────────
required_pax_keys = {"passenger_id","name","age","sex","survived","pclass",
                     "fare","embarked","family_size","is_alone",
                     "title","age_group","fare_per_person"}
first = loaded["passengers"][0]
assert required_pax_keys.issubset(first.keys()), f"Missing passenger keys: {required_pax_keys - first.keys()}"
print(f"[PASS] All required passenger fields present")

# ── Spot-check first passenger ──────────────────────────────────────────────
print(f"\nFirst passenger record:")
for k, v in first.items():
    print(f"  {k:<18}: {v}")

print("\n" + "=" * 55)
print("  All validation checks passed! ✓")
print("=" * 55)


JSON VALIDATION
[PASS] Top-level keys present: ['metadata', 'passengers']
[PASS] All required metadata keys present
[PASS] Passenger record count: 891
[PASS] All required passenger fields present

First passenger record:
  passenger_id      : 1
  name              : Braund, Mr. Owen Harris
  age               : 22.0
  sex               : male
  survived          : 0
  pclass            : 3
  fare              : 7.25
  embarked          : S
  family_size       : 2
  is_alone          : 0
  title             : Mr
  age_group         : Young Adult
  fare_per_person   : 7.25

  All validation checks passed! ✓


In [76]:
# Final summary printout
print("=" * 55)
print("  FINAL EXPORT SUMMARY")
print("=" * 55)
meta = loaded["metadata"]
print(f"  Dataset name    : {meta['dataset_name']}")
print(f"  Export date     : {meta['export_date']}")
print(f"  Total passengers: {meta['total_passengers']}")
print(f"  Survived        : {meta['survived']}  ({meta['survival_rate_pct']}%)")
print(f"  Did not survive : {meta['did_not_survive']}")
print(f"  Average age     : {meta['average_age']}")
print(f"  Average fare    : £{meta['average_fare']}")
print(f"  Missing value cols: {list(meta['missing_values'].keys())}")
print(f"  Engineered features: {meta['engineered_features']}")
print(f"  JSON file size  : {Path(JSON_FILE).stat().st_size / 1024:.1f} KB")


  FINAL EXPORT SUMMARY
  Dataset name    : Titanic Passenger Dataset
  Export date     : 2026-04-22T12:14:46.090107
  Total passengers: 891
  Survived        : 342  (38.38%)
  Did not survive : 549
  Average age     : 29.7
  Average fare    : £32.2
  Missing value cols: ['Age', 'Cabin', 'Embarked']
  Engineered features: ['FamilySize', 'IsAlone', 'AgeGroup', 'Title', 'FarePerPerson']
  JSON file size  : 296.8 KB
